# 사용자 지정 컨테이너 + 명령줄

| 정보 | 세부 내용 |
|---|---|
| 튜토리얼 | 사용자 지정 컨테이너(Node.js)를 사용하는 Harness + ExecuteCommand |
| SDK | boto3 |
| 모델 | Claude Haiku 4.5 (Bedrock) |

**학습 내용:**
- 에이전트를 자체 환경에서 실행하도록 Harness에 **사용자 지정 컨테이너 이미지**를 연결하는 방법
- **ExecuteCommand**(`invoke_agent_runtime_command`)를 사용하여 에이전트 VM에서 명령형 명령을 직접 실행하는 방법
- 에이전트가 컨테이너 런타임(Node.js)을 사용하여 코드를 작성하고 실행할 수 있는지 확인하는 방법

**사용자 지정 컨테이너가 필요한 이유**
기본적으로 Harness 세션은 Python이 포함된 Amazon Linux 2023에서 실행됩니다. 하지만 실제 에이전트에는 특정 런타임, 시스템 라이브러리 또는 사전 설치된 종속성이 필요한 경우가 많습니다. 사용자 지정 컨테이너를 사용하면 Node.js, Go, Rust, Java 등 컨테이너 이미지로 패키징할 수 있는 자체 환경을 가져올 수 있습니다.

**사전 요구 사항:**
- Amazon Bedrock AgentCore에 접근할 수 있는 AWS 계정
- 자격 증명이 설정된 AWS CLI v2
- Python 3.10+

## 0단계: 설정

헬퍼를 가져오고 IAM 실행 역할을 생성합니다.

In [ ]:
import sys
import time
import uuid
from pathlib import Path
import boto3

# 헬퍼
sys.path.insert(0, str(Path.cwd().parent.parent))

# --- 설정 ---
from helper.iam import create_harness_role, delete_harness_role
from helper.client import get_agentcore_control_client, get_agentcore_client

# --- boto3 클라이언트 생성 ---
control = get_agentcore_control_client()
client = get_agentcore_client()

account_id = boto3.client("sts").get_caller_identity()["Account"]
print(f"Account: {account_id}")

In [ ]:
role_arn = create_harness_role()
print(f"\nExecution Role ARN: {role_arn}")

print("Waiting for IAM role to propagate...")
time.sleep(10)
print("Ready!")

## 1단계: Harness 생성

먼저 기본 Harness를 생성한 다음, 다음 단계에서 사용자 지정 컨테이너를 연결합니다. 기본값으로 시작한 뒤 사용자 지정하는 일반적인 워크플로를 두 단계로 보여 줍니다.

In [ ]:
HARNESS_NAME = f"NodeContainer_{uuid.uuid4().hex[:8]}"

resp = control.create_harness(
    harnessName=HARNESS_NAME,
    executionRoleArn=role_arn,
)

harness = resp["harness"]
harness_id = harness["harnessId"]
harness_arn = harness["arn"]
print(f"Harness ID: {harness_id}")

for i in range(12):
    resp = control.get_harness(harnessId=harness_id)
    status = resp["harness"]["status"]
    print(f"Attempt {i + 1}: {status}")
    if status == "READY":
        print("✅ Harness is ready")
        break
    time.sleep(5)

## 2단계: 사용자 지정 컨테이너 연결

퍼블릭 ECR의 **Node.js** 컨테이너 이미지를 사용하도록 Harness를 업데이트합니다. 기본 Amazon Linux 환경이 Node.js 런타임으로 대체되어 에이전트에서 `node`, `npm` 및 전체 Node.js 생태계를 사용할 수 있습니다.

퍼블릭 또는 프라이빗 ECR 이미지를 사용할 수 있습니다.
- `public.ecr.aws/docker/library/node:slim` — Node.js
- `public.ecr.aws/docker/library/python:3.12-slim` — Python
- `public.ecr.aws/docker/library/golang:1.24` — Go
- 사용자 지정 종속성이 포함된 자체 프라이빗 ECR 이미지

또한 Node.js를 사용할 수 있다고 에이전트에 알려 주는 시스템 프롬프트를 설정합니다.

In [ ]:
CONTAINER_URI = "public.ecr.aws/docker/library/node:slim"

control.update_harness(
    harnessId=harness_id,
    environmentArtifact={"optionalValue": {"containerConfiguration": {"containerUri": CONTAINER_URI}}},
    systemPrompt=[
        {
            "text": "You are a helpful coding assistant. You have access to a Node.js runtime. When asked to write and run code, save it to a file and execute it using the shell."
        }
    ],
)

print(f"Container: {CONTAINER_URI}")
print("Waiting for update...")
for i in range(24):
    resp = control.get_harness(harnessId=harness_id)
    status = resp["harness"]["status"]
    print(f"Attempt {i + 1}: {status}")
    if status == "READY":
        print("✅ Harness updated")
        break
    time.sleep(5)

## 3단계: 에이전트 호출 - Node.js 코드 작성 및 실행

이제 에이전트를 호출하여 Node.js 스크립트를 작성하고 파일로 저장한 뒤 실행하도록 요청합니다. 에이전트는 기본 제공 `file_operations` 및 `shell` 도구를 사용하며, 사용자 지정 컨테이너 덕분에 Node.js 런타임을 사용할 수 있습니다.

> 컨테이너가 변경되었으므로 에이전트가 업데이트된 환경의 새 VM을 사용하도록 **새 세션 ID**를 사용합니다.

In [ ]:
import uuid

session_id = str(uuid.uuid4()).upper()
print(f"Session ID: {session_id}\n")

response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": (
                        "Write a Node.js script that creates a simple HTTP server on port 3000 "
                        "that returns JSON with the current time, Node.js version, and platform info. "
                        "Save it to /tmp/server.js. Then test it — start the server in the "
                        "background, make an HTTP request using Node.js http module (curl is not available), "
                        "and kill the server. Show me the output."
                    )
                }
            ],
        }
    ],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
)


for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

## 4단계: VM에서 직접 명령 실행(ExecuteCommand)

`invoke_agent_runtime_command` API를 사용하면 에이전트 루프를 거치지 않고 에이전트 VM에서 **명령형 명령**을 직접 실행할 수 있습니다. 다음 작업에 유용합니다.
- 환경 점검(OS, 런타임, 설치된 패키지)
- 에이전트가 생성한 파일 읽기
- 결정론적 스크립트 실행 또는 결과물 검증

> **왜 `!pwd`를 사용하지 않나요?** Jupyter의 `!`는 **로컬 시스템**에서 명령을 실행합니다. 여기서는 AWS에 있는 **에이전트의 원격 VM**에서 명령을 실행합니다.

In [ ]:
def run_command(command: str):
    """에이전트 VM에서 명령을 실행하고 출력을 표시합니다."""
    print(f"$ {command}")
    resp = client.invoke_agent_runtime_command(
        agentRuntimeArn=harness_arn,
        runtimeSessionId=session_id,
        body={"command": command},
    )
    for event in resp["stream"]:
        if "chunk" in event:
            chunk = event["chunk"]
            if "contentDelta" in chunk:
                d = chunk["contentDelta"]
                if "stdout" in d:
                    print(d["stdout"], end="", flush=True)
                if "stderr" in d:
                    print(d["stderr"], end="", flush=True)
            elif "contentStop" in chunk:
                print(f"\n[exit: {chunk['contentStop']['exitCode']}]")
    print()

### Node.js 환경 확인

VM에서 사용자 지정 컨테이너의 런타임을 사용할 수 있는지 확인합니다.

In [ ]:
run_command("node --version")
run_command("npm --version")

### 에이전트가 생성한 코드 점검

에이전트가 작성한 스크립트를 다시 읽습니다.

In [ ]:
run_command("cat /tmp/server.js")

### ExecuteCommand로 직접 스크립트 실행

에이전트 루프를 호출하지 않고 에이전트의 코드를 직접 다시 실행할 수도 있습니다.

In [ ]:
# 서버를 시작하고 Node.js http 모듈로 테스트한 다음 중지
run_command(
    "node /tmp/server.js & sleep 1 && node -e \"require('http').get('http://localhost:3000',r=>{let d='';r.on('data',c=>d+=c);r.on('end',()=>console.log(JSON.stringify(JSON.parse(d),null,2)))})\" && kill %1 2>/dev/null"
)

### VM 환경 살펴보기

실행 중인 항목을 파악하는 데 유용한 명령을 몇 가지 더 살펴봅니다.

In [ ]:
run_command("cat /etc/os-release")
run_command("pwd")
run_command("whoami")
run_command("ls -la /tmp/")

## 5단계: npm 패키지 설치 및 사용

완전한 Node.js 환경이 있으므로 에이전트는 런타임에 npm 패키지를 설치할 수도 있습니다. 패키지를 설치하고 사용하도록 요청해 보겠습니다.

In [ ]:
# 같은 세션에서는 VM 상태가 유지됨
response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": (
                        "Install the 'chalk' npm package (latest), then write a Node.js script at /tmp/colors.js "
                        "that uses chalk to print a colorful welcome banner with different colored lines. Run it."
                    )
                }
            ],
        }
    ],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
)

for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

## 리소스 정리

테스트를 마치면 **이 셀들을 실행**하여 모든 리소스를 삭제하고 요금이 발생하지 않도록 합니다.

In [ ]:
control.delete_harness(harnessId=harness_id)
print(f"Deleted harness: {harness_id}")

In [ ]:
# IAM 역할 삭제(생성한 경우, 선택 사항)
delete_harness_role()